# Compare three bi-objective Pareto methods

This example applies the normalised weighted-sum, augmented epsilon-constraint, and augmented weighted Tchebycheff sweeps to the same copy of Calliope's documented national-scale model. The final figure overlays all three fronts.

## 1. Locate the repository, example model, and solver

In [1]:
import os
import shutil
import sys
from pathlib import Path

repository_root = next(
    (
        candidate
        for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (candidate / "src" / "calliope" / "multiobjective").is_dir()
    ),
    None,
)
if repository_root is None:
    raise RuntimeError("Run this notebook from within the Calliope repository.")

local_source = repository_root / "src"
MODEL_PATH = (
    local_source
    / "calliope"
    / "multiobjective"
    / "examples"
    / "national_scale"
    / "model.yaml"
)
sys.path.insert(0, str(local_source))

kernel_bin = Path(sys.executable).parent
os.environ["PATH"] = f"{kernel_bin}{os.pathsep}{os.environ['PATH']}"

import calliope

CBC = shutil.which("cbc")
if CBC is None:
    raise RuntimeError("CBC was not found in the selected notebook kernel.")

print(f"Python:   {sys.executable}")
print(f"Calliope: {calliope.__file__}")
print(f"Model:    {MODEL_PATH}")
print(f"CBC:      {CBC}")

Python:   /home/mikhailavrutskii/miniconda3/envs/calliope-dashboard-070/bin/python
Calliope: /home/mikhailavrutskii/PycharmProjects/calliope-research/thd-spatial-ai-calliope/src/calliope/__init__.py
Model:    /home/mikhailavrutskii/PycharmProjects/calliope-research/thd-spatial-ai-calliope/src/calliope/multiobjective/examples/national_scale/model.yaml
CBC:      /home/mikhailavrutskii/miniconda3/envs/calliope-dashboard-070/bin/cbc


## 2. Define one study shared by all methods

The factory returns a fresh model for every method. The override ensures that both cost classes exist in Calliope's mutable objective-weight input before the backend is built.

In [2]:
from IPython.display import display
from calliope.multiobjective import (
    AugmentedEpsilonConstraint,
    AugmentedTchebycheffSweep,
    Objective,
    ParetoStudy,
    WeightedSumSweep,
)

calliope.set_log_verbosity("WARNING", include_solver_output=False)

OBJECTIVE_WEIGHT_OVERRIDE = {
    "data_definitions": {
        "objective_cost_weights": {
            "data": [1, 0],
            "index": ["monetary", "emissions"],
            "dims": "costs",
        }
    }
}


def national_scale_model():
    return calliope.read_yaml(
        MODEL_PATH,
        scenario="minimize_emissions_costs",
        override_dict=OBJECTIVE_WEIGHT_OVERRIDE,
    )


study = ParetoStudy(
    model_factory=national_scale_model,
    objective_1=Objective(
        "monetary", label="Total system cost", unit="EUR"
    ),
    objective_2=Objective(
        "emissions", label="CO2 emissions", unit="kg CO2"
    ),
    solve_options={
        "solver": "cbc",
        "solver_options": {
            "primalTolerance": 1e-10,
            "dualTolerance": 1e-10,
        },
    },
)

study

ParetoStudy(model_factory=<function national_scale_model at 0x7494d02e36d0>, objective_1=Objective(cost_class='monetary', label='Total system cost', unit='EUR'), objective_2=Objective(cost_class='emissions', label='CO2 emissions', unit='kg CO2'), build_options={}, solve_options={'solver': 'cbc', 'solver_options': {'primalTolerance': 1e-10, 'dualTolerance': 1e-10}})

## 3. Generate all three fronts

Twenty requested points keep the example readable. Increase `N_POINTS` for a denser comparison.

In [3]:
import pandas as pd

N_POINTS = 20
AUGMENTATION = 1e-6

methods = {
    "Weighted sum": WeightedSumSweep(points=N_POINTS),
    "Augmented epsilon-constraint": AugmentedEpsilonConstraint(
        points=N_POINTS, augmentation=AUGMENTATION
    ),
    "Augmented Tchebycheff": AugmentedTchebycheffSweep(
        points=N_POINTS, augmentation=AUGMENTATION
    ),
}

results = {name: study.run(method) for name, method in methods.items()}

summary = pd.DataFrame(
    [
        {
            "method": name,
            "requested_points": len(result.points),
            "unique_points": len(
                result.points.drop_duplicates(["objective_1", "objective_2"])
            ),
        }
        for name, result in results.items()
    ]
)
summary

,method,requested_points,unique_points
0,Weighted sum,20,17
1,Augmented epsilon-constraint,20,20
2,Augmented Tchebycheff,20,20


## 4. Overlay the Pareto fronts

Exactly coincident points from one method are grouped. Hover over a marker to inspect its method parameters and multiplicity.

In [8]:
import plotly.express as px

plot_frames = []
for method_name, result in results.items():
    points = result.points.copy()
    parameter_columns = [
        column
        for column in points.columns
        if column not in {"point_id", "objective_1", "objective_2"}
    ]
    points["parameters"] = points.apply(
        lambda row: "<br>".join(
            f"{column}={row[column]:.6g}" for column in parameter_columns
        ),
        axis=1,
    )
    grouped = (
        points.groupby(["objective_1", "objective_2"], as_index=False)
        .agg(
            point_ids=("point_id", lambda values: ", ".join(map(str, values))),
            parameters=("parameters", "<br>".join),
            coincident_points=("point_id", "size"),
        )
        .sort_values("objective_1")
    )
    grouped["method"] = method_name
    plot_frames.append(grouped)

plot_points = pd.concat(plot_frames, ignore_index=True)
figure = px.line(
    plot_points,
    x="objective_1",
    y="objective_2",
    color="method",
    markers=True,
    custom_data=["point_ids", "parameters", "coincident_points"],
    labels={
        "objective_1": "Total system cost [EUR]",
        "objective_2": "CO2 emissions [kg CO2]",
        "method": "Method",
    },
    title=f"Pareto-front comparison — {N_POINTS} requested points per method",
)
figure.update_traces(
    hovertemplate=(
        "Total system cost: %{x:,.4g}<br>"
        "CO2 emissions: %{y:,.4g}<br>"
        "Point IDs: %{customdata[0]}<br>"
        "Parameters:<br>%{customdata[1]}<br>"
        "Coincident points: %{customdata[2]}"
        "<extra>%{fullData.name}</extra>"
    )
)
figure

## 5. Inspect one complete Calliope solution

In [5]:
SELECTED_METHOD = "Augmented Tchebycheff"
SELECTED_POINT_ID = N_POINTS // 2

selected_result = results[SELECTED_METHOD]
selected_point = selected_result.points.loc[
    selected_result.points["point_id"] == SELECTED_POINT_ID
].iloc[0]
selected_solution = selected_result.solution(SELECTED_POINT_ID)

display(selected_point.to_frame("value"))
selected_solution["flow_cap"].fillna(0).to_series().loc[lambda values: values > 1e-8].sort_values(ascending=False)

,value
point_id,1.000000e+01
weight_1,5.263158e-01
weight_2,4.736842e-01
objective_1,5.765255e+05
objective_2,2.160168e+08


nodes      techs                 carriers
region1    demand_power          power       39033.5200
           ccgt                  power       20900.6640
region1_1  csp                   power       10000.0000
region1_3  csp                   power       10000.0000
region1    region1_to_region1_1  power        9000.0000
region1_1  region1_to_region1_1  power        9000.0000
region1    region1_to_region1_3  power        9000.0000
region1_3  region1_to_region1_3  power        9000.0000
region1    region1_to_region2    power        3278.6894
region2    region1_to_region2    power        3278.6894
           demand_power          power        2909.8360
region1_2  csp                   power        2644.1360
region1    region1_to_region1_2  power        2379.7224
region1_2  region1_to_region1_2  power        2379.7224
region2    battery               power        1000.0000
Name: flow_cap, dtype: float64